### Goal


We do the model training on the following time series models for the short-term prediction:
- [Holt-Winters (Triple Exponential Smoothing)](https://www.statsmodels.org/devel/generated/statsmodels.tsa.holtwinters.ExponentialSmoothing.html)
- [SARIMA](https://www.statsmodels.org/devel/generated/statsmodels.tsa.arima.model.ARIMA.html#statsmodels.tsa.arima.model.ARIMA)
- [Facebook Prophet](http://facebook.github.io/prophet/)

Our selection of the models for tuning and testing are as follows:

| Models | Y/N |Comment |
| ---------- | --  | -----------  |
| Dummy  | ✔️   | |
| Exponential Smoothing (non-CV)| ✔️ | Use manual smoothing |
| Exponential Smoothing (CV) | ✔️ | Use auto smoothing |
| Auto_SRIMA | ❌ | Abandoned because of the runtime issue | 
| Manual_SRIMA | ✔️ | Use manual seasonal order |
| Prophet | ✔️ | |
| Prophet (full) | ✔️ | Trained on the full data set from 01/01/2001 |

### Data processing and set-up


We process the data and set up the required packages.

In [4]:
import pandas, numpy, matplotlib, seaborn, sklearn, statsmodels, prophet
print("All packages imported successfully!")

All packages imported successfully!


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from datetime import datetime, timedelta
from seaborn import set_style
from sklearn.metrics import mean_squared_error

set_style("whitegrid")

We load the data from `data/arxiv-totals.parquet` and set up our training data set from 01/01/2001 (Monday) to 03/14/2025 (Monday), and testing data set to 03/17/2025 (Monday).

In [6]:
df = pd.read_parquet("../data/arxiv-totals.parquet")

df_train = df[
    (df.index >= pd.Timestamp(2001, 1, 1)) & (df.index <= pd.Timestamp(2025, 3, 14))
]
df_test = df[df.index >= pd.Timestamp(2025, 3, 17)]

In [7]:
print(df.columns)
print(df_train.shape, df_test.shape)
print(df.head())

Index(['hep-th', 'physics.pop-ph', 'math.LO', 'math.FA', 'math.MG', 'cs.CC',
       'math.CO', 'math.PR', 'math.DS', 'cs.GR',
       ...
       'econ.EM', 'stat.CO', 'stat.OT', 'q-fin.EC', 'eess.SY', 'econ.GN',
       'eess.AS', 'eess.IV', 'eess.SP', 'q-fin.MF'],
      dtype='object', length=158)
(6315, 158) (20, 158)
            hep-th  physics.pop-ph  math.LO  math.FA  math.MG  cs.CC  math.CO  \
date                                                                            
1986-04-28     1.0             1.0      0.0      0.0      0.0    0.0      0.0   
1988-11-14     1.0             0.0      0.0      0.0      0.0    0.0      0.0   
1989-04-17     0.0             0.0      1.0      0.0      0.0    0.0      0.0   
1989-10-27     0.0             0.0      0.0      3.0      3.0    0.0      0.0   
1989-11-10     0.0             0.0      0.0      1.0      1.0    0.0      0.0   

            math.PR  math.DS  cs.GR  ...  econ.EM  stat.CO  stat.OT  q-fin.EC  \
date                           

<!-- This indicates that the time series has seasonality, with season of a week. Other categories exhibit similar correlograms, so effective models should likely take weekly seasonality into account (notice that the seasonal parameter should be 5 instead of 7 since the papers are only submitted on business days). Looking at the graphs, there is also a global trend to take into account. -->

We will use [statsmodels](https://www.statsmodels.org/stable/index.html) as our choice of time series library (Install the module `statsmodels` by using Anaconda `conda install -c conda-forge statsmodels`). In partucular, see [Time Series analysis `tsa`](https://www.statsmodels.org/devel/tsa.html).

In [8]:
## Importing statsmodels to check that we have it installed
import statsmodels as sm

In [9]:
## printing the statsmodels version
print(sm.__version__)

0.14.4


### Cross-validation set-up

First we prepare a 5-fold validation. We take a gap of 5 business days between train and validation splits, and we take a test size of 15 business days (note that our forecasting horizon is 5 business days).

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

# Prepare CV splits
ts_cv = TimeSeriesSplit(n_splits=5, test_size=15, max_train_size=150)
splits = [(train_idx, test_idx) for train_idx, test_idx in ts_cv.split(df_train)]

# Prepare CV splits with full data for prophet_full
ts_cv_full = TimeSeriesSplit(n_splits=5, test_size=15)
splits_full= [(train_idx, test_idx) for train_idx, test_idx in ts_cv_full.split(df_train)]

### EST

Our grid search for smoothing parameters of EST is selected as the following:

| Config | α (level) | β (trend) | γ (seasonal) | Intuition                                                                 |
| ------ | --------- | --------- | ------------ | ------------------------------------------------------------------------- |
| #1     | 0.1       | 0.1       | 0.1          | Very **stable** model — slow updates to all components                    |
| #2     | 0.1       | 0.1       | 0.8          | Keeps **level/trend stable**, allows **seasonal** pattern to vary quickly |
| #3     | 0.3       | 0.1       | 0.3          | Slightly more responsive level & seasonality, but trend still stable      |
| #4     | 0.5       | 0.3       | 0.5          | **Balanced responsiveness** — a middle ground, adapts fairly quickly      |


In [16]:
# Define Exponential Smoothing structures and smoothing parameters
est_structures = [
    {"trend": None, "seasonal": "add"},
    {"trend": None, "seasonal": "mul"},
    {"trend": "add", "seasonal": "add"},
    {"trend": "add", "seasonal": "mul"}
]
smoothing_grid = [
    {"level": 0.1, "trend": 0.1, "seasonal": 0.1},
    {"level": 0.1, "trend": 0.1, "seasonal": 0.8},
    {"level": 0.3, "trend": 0.1, "seasonal": 0.3},
    {"level": 0.5, "trend": 0.3, "seasonal": 0.5},
]

### SARIMA

Our grid search for seasonal order of the manual SARIMA is selected as the following:
| Seasonal Order (P,D,Q,s) | Description                                        | Intuition                                                         |
|--------------------------|----------------------------------------------------|----------------------------------------------------------------------|
| (0,1,0,5)                | Seasonal differencing only                         | Baseline seasonal differencing, no seasonal AR/MA                    |
| (1,1,0,5)                | Add seasonal AR(1)                                 | Capture short-term seasonal autocorrelation                          |
| (0,1,1,5)                | Add seasonal MA(1)                                 | Capture short-term seasonal shocks                                   |
| (1,1,1,5)                | Add both seasonal AR(1) and MA(1)                  | Capture both autocorrelation and shocks at seasonal frequency        |


In [17]:
# Define SARIMA parameter sets for grid search
sarima_param_sets = [(0,1,0,5), (1,1,0,5), (0,1,1,5), (1,1,1,5)]

### Prophet

In [ ]:
# Install cmdstanpy to fix Prophet optimization runtime issues
import cmdstanpy
cmdstanpy.install_cmdstan()
cmdstanpy.install_cmdstan(compiler=True) # only valid on Windows

prophet_rmses = np.zeros(5)

### Model tuning

In [38]:
from statsmodels.tsa.api import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error
from prophet import Prophet
import pandas as pd
import numpy as np
import json
import warnings

warnings.filterwarnings("ignore")

# Load categories
with open("../data/arxiv-categories.json", "r") as f:
    arxiv_categories_descriptions = json.load(f)
categories = sorted([cat["tag"] for cat in arxiv_categories_descriptions])
# categories = sorted(
#     [cat["tag"] for cat in arxiv_categories_descriptions]
#     # We exclude ["q-bio", "cond-mat", "astro-ph"] because they disappeared before our sample starting date.
# )

# Initialize result containers
cv_results_dict = {}
cv_best_params_dict = {}

# Ensure business day frequency
df_train = df_train.asfreq("B")  

# Iterate over each category and perform time series cross-validation
print(f"Number of categories: {len(categories)}")
print(f"Number of splits: {len(splits)}")
print(f"Number of training samples: {len(df_train)}")
print(f"Number of test samples: {len(df_test)}")

for i, category in enumerate(categories, 1):

    print(f"[{i}/{len(categories)}] Tuning category: {category}")

    # Prepare data for the current category
    cv_best_params = {}
    cv_results = {}

    
    # Dummy model
    dummy_rmses = []
    for train_idx, test_idx in splits:
        y_train = df_train.iloc[train_idx][category].fillna(0)
        y_test = df_train.iloc[test_idx][category].fillna(0)
        dummy_preds = np.full_like(y_test, y_train.mean())
        rmse = np.sqrt(mean_squared_error(y_test, dummy_preds))
        dummy_rmses.append(rmse)
    train_mean = df_train[category].fillna(0).mean()
    cv_results["Dummy"] = np.nanmean(dummy_rmses) / train_mean
    cv_best_params["Dummy"] = {"value": y_train.mean()}


    # EST_NCV (manual smoothing)
    best_est_ncv_rmse = np.inf
    best_est_ncv_config = None
    for struct in est_structures:
        for smooth in smoothing_grid:
            rmses = []
            for train_idx, test_idx in splits:
                y_train = df_train.iloc[train_idx][category].fillna(0)
                y_test = df_train.iloc[test_idx][category].fillna(0)
                try:
                    model = ExponentialSmoothing(
                        y_train,
                        trend=struct["trend"],
                        seasonal=struct["seasonal"],
                        seasonal_periods=5
                    ).fit(
                        smoothing_level=smooth["level"],
                        smoothing_trend=smooth["trend"],
                        smoothing_seasonal=smooth["seasonal"],
                        optimized=False
                    )
                    preds = model.forecast(len(y_test))
                    rmse = np.sqrt(mean_squared_error(y_test, preds))
                    rmses.append(rmse)
                except:
                    rmses.append(np.nan)
            avg_rmse = np.nanmean(rmses)
            if avg_rmse < best_est_ncv_rmse:
                best_est_ncv_rmse = avg_rmse
                best_est_ncv_config = {
                    "trend": struct["trend"],
                    "seasonal": struct["seasonal"],
                    "smoothing_level": smooth["level"],
                    "smoothing_trend": smooth["trend"],
                    "smoothing_seasonal": smooth["seasonal"]
                }
    cv_results["EST_NCV"] = best_est_ncv_rmse / train_mean
    cv_best_params["EST_NCV"] = best_est_ncv_config


    # EST_CV (auto smoothing)
    best_est_cv_rmse = np.inf
    best_est_cv_config = None
    for struct in est_structures:
        rmses = []
        for train_idx, test_idx in splits:
            y_train = df_train.iloc[train_idx][category].fillna(0)
            y_test = df_train.iloc[test_idx][category].fillna(0)
            try:
                model = ExponentialSmoothing(
                    y_train,
                    trend=struct["trend"],
                    seasonal=struct["seasonal"],
                    seasonal_periods=5
                ).fit(optimized=True)
                preds = model.forecast(len(y_test))
                rmse = np.sqrt(mean_squared_error(y_test, preds))
                rmses.append(rmse)
            except:
                rmses.append(np.nan)
        avg_rmse = np.nanmean(rmses)
        if avg_rmse < best_est_cv_rmse:
            best_est_cv_rmse = avg_rmse
            best_est_cv_config = {
                "trend": struct["trend"],
                "seasonal": struct["seasonal"]
            }
    cv_results["EST_CV"] = best_est_cv_rmse / train_mean
    cv_best_params["EST_CV"] = best_est_cv_config


    # SARIMA_CV (manual seasonal order)
    best_sarima_rmse = np.inf
    best_sarima_config = None
    for seasonal_order in sarima_param_sets:
        rmses = []
        for train_idx, test_idx in splits:
            y_train = df_train.iloc[train_idx][category].fillna(0)
            y_test = df_train.iloc[test_idx][category].fillna(0)
            try:
                model = ARIMA(y_train, order=(0, 0, 0), seasonal_order=seasonal_order)
                fitted = model.fit()
                preds = fitted.forecast(steps=len(y_test))
                rmse = np.sqrt(mean_squared_error(y_test, preds))
                rmses.append(rmse)
            except:
                rmses.append(np.nan)
        avg_rmse = np.nanmean(rmses)
        if avg_rmse < best_sarima_rmse:
            best_sarima_rmse = avg_rmse
            best_sarima_config = {"seasonal_order": seasonal_order}
    cv_results["SARIMA_CV"] = best_sarima_rmse / train_mean
    cv_best_params["SARIMA_CV"] = best_sarima_config


    # Prophet
    prophet_rmses = []
    prophet_params = {
        "seasonality_mode": "additive", # or "multiplicative"
        "weekly_seasonality": False,
        "yearly_seasonality": False,
        # "custom_period": 5,
        # "fourier_order": 3
        "changepoint_prior_scale": 0.05,  # optional: restrict overfitting
    }
    for fold, (train_index, test_index) in enumerate(splits):
        df_tt = df_train.iloc[train_index].reset_index()
        df_holdout = df_train.iloc[test_index].reset_index()
        
        y_fold= df_tt[category].fillna(0)

        # # Optional: skip Prophet if data quality is too low
        # if y_fold.nunique() < 10 or (y_fold <= 0).all():
        #     raise ValueError("Too few unique values or only zero/negative entries")
        # y_fold= df_tt[category].fillna(0)

        # Ensure date column is in datetime format
        df_tt["date"] = pd.to_datetime(df_tt["date"])
        df_holdout["date"] = pd.to_datetime(df_holdout["date"])

        prophet = Prophet(
            seasonality_mode=prophet_params["seasonality_mode"],
            weekly_seasonality=prophet_params["weekly_seasonality"],
            yearly_seasonality=prophet_params["yearly_seasonality"],
            changepoint_prior_scale=prophet_params["changepoint_prior_scale"]
        )
        # Prepare dataframes for Prophet
        prophet_tt = df_tt[["date", category]].rename(columns={"date": "ds", category: "y"})
        prophet_holdout = df_holdout[["date", category]].rename(columns={"date": "ds", category: "y"})
        # Optional: add custom seasonality
        # prophet.add_seasonality(
        #     name="custom",
        #     period=prophet_params["custom_period"],
        #     fourier_order=prophet_params["fourier_order"]
        # )
        # Fit Prophet model
        prophet.fit(prophet_tt)
        forecast = prophet.predict(prophet_holdout[["ds"]])
        preds = forecast["yhat"].values
        # Calculate RMSE for Prophet model
        rmse = np.sqrt(mean_squared_error(prophet_holdout["y"], preds))
        prophet_rmses.append(rmse)
    # Average Prophet RMSE over folds
    avg_prophet_rmse = np.nanmean(prophet_rmses)
    cv_results["Prophet"] = avg_prophet_rmse / train_mean if not np.isnan(avg_prophet_rmse) else np.nan
    cv_best_params["Prophet"] = prophet_params


    # Prophet Full (using full training set > 150)
    prophet_full_rmses = []
    prophet_full_params = {
        "seasonality_mode": "additive", # or "multiplicative"
        "weekly_seasonality": False,
        "yearly_seasonality": False,
        # "custom_period": 5,
        # "fourier_order": 3
        "changepoint_prior_scale": 0.05,  # optional: restrict overfitting
    }
    for fold, (train_index, test_index) in enumerate(splits_full):
        df_tt = df_train.iloc[train_index].reset_index()
        df_holdout = df_train.iloc[test_index].reset_index()
        
        y_fold= df_tt[category].fillna(0)

        # # Optional: skip Prophet if data quality is too low
        # if y_fold.nunique() < 10 or (y_fold <= 0).all():
        #     raise ValueError("Too few unique values or only zero/negative entries")
        # y_fold= df_tt[category].fillna(0)

        # Ensure date column is in datetime format
        df_tt["date"] = pd.to_datetime(df_tt["date"])
        df_holdout["date"] = pd.to_datetime(df_holdout["date"])

        prophet = Prophet(
            seasonality_mode=prophet_params["seasonality_mode"],
            weekly_seasonality=prophet_params["weekly_seasonality"],
            yearly_seasonality=prophet_params["yearly_seasonality"],
            changepoint_prior_scale=prophet_params["changepoint_prior_scale"]
        )
        # Prepare dataframes for Prophet
        prophet_tt = df_tt[["date", category]].rename(columns={"date": "ds", category: "y"})
        prophet_holdout = df_holdout[["date", category]].rename(columns={"date": "ds", category: "y"})
        # Optional: add custom seasonality
        # prophet.add_seasonality(
        #     name="custom",
        #     period=prophet_params["custom_period"],
        #     fourier_order=prophet_params["fourier_order"]
        # )
        # Fit Prophet model
        prophet.fit(prophet_tt)
        forecast = prophet.predict(prophet_holdout[["ds"]])
        preds = forecast["yhat"].values
        # Calculate RMSE for Prophet model
        rmse = np.sqrt(mean_squared_error(prophet_holdout["y"], preds))
        prophet_full_rmses.append(rmse)
    # Average Prophet RMSE over folds
    avg_prophet_full_rmse = np.nanmean(prophet_full_rmses)
    cv_results["Prophet_Full"] = avg_prophet_full_rmse / train_mean if not np.isnan(avg_prophet_rmse) else np.nan
    cv_best_params["Prophet_Full"] = prophet_full_params

    # Store all results
    cv_results_dict[category] = cv_results
    cv_best_params_dict[category] = cv_best_params


Number of categories: 155
Number of splits: 5
Number of training samples: 6315
Number of test samples: 20
[1/155] Tuning category: astro-ph.CO


13:42:41 - cmdstanpy - INFO - Chain [1] start processing
13:42:41 - cmdstanpy - INFO - Chain [1] done processing
13:42:42 - cmdstanpy - INFO - Chain [1] start processing
13:42:42 - cmdstanpy - INFO - Chain [1] done processing
13:42:42 - cmdstanpy - INFO - Chain [1] start processing
13:42:42 - cmdstanpy - INFO - Chain [1] done processing
13:42:42 - cmdstanpy - INFO - Chain [1] start processing
13:42:42 - cmdstanpy - INFO - Chain [1] done processing
13:42:42 - cmdstanpy - INFO - Chain [1] start processing
13:42:42 - cmdstanpy - INFO - Chain [1] done processing
13:42:43 - cmdstanpy - INFO - Chain [1] start processing
13:42:44 - cmdstanpy - INFO - Chain [1] done processing
13:42:44 - cmdstanpy - INFO - Chain [1] start processing
13:42:45 - cmdstanpy - INFO - Chain [1] done processing
13:42:45 - cmdstanpy - INFO - Chain [1] start processing
13:42:46 - cmdstanpy - INFO - Chain [1] done processing
13:42:46 - cmdstanpy - INFO - Chain [1] start processing
13:42:48 - cmdstanpy - INFO - Chain [1]

[2/155] Tuning category: astro-ph.EP


13:42:52 - cmdstanpy - INFO - Chain [1] start processing
13:42:52 - cmdstanpy - INFO - Chain [1] done processing
13:42:52 - cmdstanpy - INFO - Chain [1] start processing
13:42:52 - cmdstanpy - INFO - Chain [1] done processing
13:42:52 - cmdstanpy - INFO - Chain [1] start processing
13:42:53 - cmdstanpy - INFO - Chain [1] done processing
13:42:53 - cmdstanpy - INFO - Chain [1] start processing
13:42:53 - cmdstanpy - INFO - Chain [1] done processing
13:42:53 - cmdstanpy - INFO - Chain [1] start processing
13:42:53 - cmdstanpy - INFO - Chain [1] done processing
13:42:53 - cmdstanpy - INFO - Chain [1] start processing
13:42:54 - cmdstanpy - INFO - Chain [1] done processing
13:42:54 - cmdstanpy - INFO - Chain [1] start processing
13:42:55 - cmdstanpy - INFO - Chain [1] done processing
13:42:55 - cmdstanpy - INFO - Chain [1] start processing
13:42:56 - cmdstanpy - INFO - Chain [1] done processing
13:42:56 - cmdstanpy - INFO - Chain [1] start processing
13:42:57 - cmdstanpy - INFO - Chain [1]

[3/155] Tuning category: astro-ph.GA


...

14:04:55 - cmdstanpy - INFO - Chain [1] start processing
14:04:55 - cmdstanpy - INFO - Chain [1] done processing
14:04:55 - cmdstanpy - INFO - Chain [1] start processing
14:04:55 - cmdstanpy - INFO - Chain [1] done processing
14:04:55 - cmdstanpy - INFO - Chain [1] start processing
14:04:55 - cmdstanpy - INFO - Chain [1] done processing
14:04:56 - cmdstanpy - INFO - Chain [1] start processing
14:04:56 - cmdstanpy - INFO - Chain [1] done processing
14:04:56 - cmdstanpy - INFO - Chain [1] start processing
14:04:56 - cmdstanpy - INFO - Chain [1] done processing
14:04:57 - cmdstanpy - INFO - Chain [1] start processing
14:04:58 - cmdstanpy - INFO - Chain [1] done processing
14:04:59 - cmdstanpy - INFO - Chain [1] start processing
14:05:00 - cmdstanpy - INFO - Chain [1] done processing
14:05:00 - cmdstanpy - INFO - Chain [1] start processing
14:05:02 - cmdstanpy - INFO - Chain [1] done processing
14:05:03 - cmdstanpy - INFO - Chain [1] start processing
14:05:05 - cmdstanpy - INFO - Chain [1]

[154/155] Tuning category: stat.OT


14:05:14 - cmdstanpy - INFO - Chain [1] start processing
14:05:14 - cmdstanpy - INFO - Chain [1] done processing
14:05:14 - cmdstanpy - INFO - Chain [1] start processing
14:05:14 - cmdstanpy - INFO - Chain [1] done processing
14:05:15 - cmdstanpy - INFO - Chain [1] start processing
14:05:15 - cmdstanpy - INFO - Chain [1] done processing
14:05:15 - cmdstanpy - INFO - Chain [1] start processing
14:05:15 - cmdstanpy - INFO - Chain [1] done processing
14:05:15 - cmdstanpy - INFO - Chain [1] start processing
14:05:15 - cmdstanpy - INFO - Chain [1] done processing
14:05:15 - cmdstanpy - INFO - Chain [1] start processing
14:05:16 - cmdstanpy - INFO - Chain [1] done processing
14:05:17 - cmdstanpy - INFO - Chain [1] start processing
14:05:17 - cmdstanpy - INFO - Chain [1] done processing
14:05:18 - cmdstanpy - INFO - Chain [1] start processing
14:05:19 - cmdstanpy - INFO - Chain [1] done processing
14:05:19 - cmdstanpy - INFO - Chain [1] start processing
14:05:19 - cmdstanpy - INFO - Chain [1]

[155/155] Tuning category: stat.TH


14:05:25 - cmdstanpy - INFO - Chain [1] start processing
14:05:25 - cmdstanpy - INFO - Chain [1] done processing
14:05:25 - cmdstanpy - INFO - Chain [1] start processing
14:05:25 - cmdstanpy - INFO - Chain [1] done processing
14:05:25 - cmdstanpy - INFO - Chain [1] start processing
14:05:25 - cmdstanpy - INFO - Chain [1] done processing
14:05:26 - cmdstanpy - INFO - Chain [1] start processing
14:05:26 - cmdstanpy - INFO - Chain [1] done processing
14:05:26 - cmdstanpy - INFO - Chain [1] start processing
14:05:26 - cmdstanpy - INFO - Chain [1] done processing
14:05:26 - cmdstanpy - INFO - Chain [1] start processing
14:05:27 - cmdstanpy - INFO - Chain [1] done processing
14:05:27 - cmdstanpy - INFO - Chain [1] start processing
14:05:27 - cmdstanpy - INFO - Chain [1] done processing
14:05:27 - cmdstanpy - INFO - Chain [1] start processing
14:05:28 - cmdstanpy - INFO - Chain [1] done processing
14:05:28 - cmdstanpy - INFO - Chain [1] start processing
14:05:29 - cmdstanpy - INFO - Chain [1]

We print out and save the CV RMSE results `cv_results_dict` and the best parameters `cv_best_params_dict` for each cateogry and each model.

In [41]:
import pandas as pd
import json

# Convert results to DataFrame
results_df = pd.DataFrame.from_dict(cv_results_dict, orient="index")

# Print results
print("📊 Cross-Validation RMSE Results (normalized by train mean):")
print(results_df.round(3))  # Rounded for readability

# Print best parameters
print("\n🔧 Best Parameters for Each Category:")
for category, model_dict in cv_best_params_dict.items():
    print(f"\nCategory: {category}")
    for model_name, params in model_dict.items():
        print(f"  {model_name}: {params}")

# Save results to CSV
results_df.to_csv("cv_results.csv")

# Save best parameters to JSON
with open("cv_best_params.json", "w") as f:
    json.dump(cv_best_params_dict, f, indent=2)

print("\n✅ Results saved to 'cv_results.csv' and 'cv_best_params.json'.")


📊 Cross-Validation RMSE Results (normalized by train mean):
             Dummy  EST_NCV  EST_CV  SARIMA_CV  Prophet  Prophet_Full
astro-ph.CO  0.511    0.562   0.570      0.466    0.543         0.505
astro-ph.EP  0.857    0.839   0.849      0.846    0.889         0.846
astro-ph.GA  0.780    0.816   0.816      0.793    0.807         0.782
astro-ph.HE  0.731    0.725   0.760      0.707    0.757         0.727
astro-ph.IM  0.979    0.959   0.931      0.929    0.977         0.951
...            ...      ...     ...        ...      ...           ...
stat.CO      1.350    1.399   1.351      1.331    1.422         1.344
stat.ME      1.267    1.070   1.075      1.002    1.275         1.252
stat.ML      1.041    0.856   0.816      0.725    1.128         1.446
stat.OT      3.075    3.364   3.084      3.097    3.109         3.100
stat.TH      0.914    0.926   0.907      0.872    0.913         0.914

[155 rows x 6 columns]

🔧 Best Parameters for Each Category:

Category: astro-ph.CO
  Dummy: {'valu